# Face Recognition using PCA (Eigenfaces) + ANN

**Algorithm:** PCA (Turk & Pentland, 1991) + Backpropagation Neural Network  
**Dataset:** AT&T/ORL Face Dataset (downloaded automatically from GitHub)  
**Split:** 60% Train / 40% Test  
**Libraries:** NumPy, SciPy, OpenCV (as per project requirements)

---
### Steps Covered
1. Download and load the face dataset
2. Build Face Database matrix (mn x p)
3. Compute Mean Face
4. Mean-Zero Faces
5. Surrogate Covariance (p x p)
6. Eigenvalue / Eigenvector decomposition
7. Generate Eigenfaces
8. Generate Face Signatures
9. Train ANN (Backpropagation from scratch using NumPy)
10. Evaluate accuracy
11. **Experiment A** — Vary k, plot Accuracy vs k
12. **Experiment B** — Imposter detection

---
## STEP 1 — Install and Import Libraries

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'opencv-python', 'scipy', 'matplotlib', 'numpy',
                'scikit-learn', '--quiet'])

import os
import zipfile
import urllib.request
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.linalg import eigh
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('All libraries imported successfully!')

---
## STEP 2 — Download and Extract Dataset

Dataset: https://github.com/robaita/introduction_to_machine_learning  
The AT&T/ORL face dataset has **40 subjects x 10 images = 400 images**  
Each image is **92 x 112 pixels** in grayscale.

In [ ]:
DATASET_URL = 'https://github.com/robaita/introduction_to_machine_learning/raw/main/dataset.zip'
ZIP_PATH    = 'dataset.zip'
IMG_SIZE    = (92, 112)  # (width, height)

# Download
if not os.path.exists(ZIP_PATH):
    print('Downloading dataset...')
    urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
    print('Download complete!')
else:
    print('ZIP already present.')

# Extract fresh every time to avoid stale data
import shutil
if os.path.exists('extracted_dataset'):
    shutil.rmtree('extracted_dataset')
print('Extracting...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('extracted_dataset')
print('Extraction complete!')

# Show extracted structure (for debugging)
print('\nExtracted folder tree:')
for root, dirs, files in os.walk('extracted_dataset'):
    dirs.sort()
    level = root.replace('extracted_dataset', '').count(os.sep)
    if level > 3: continue
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level >= 2:
        sample = [f for f in sorted(files)[:2]]
        for sf in sample:
            print(f'{indent}  {sf}')

# Smart dataset folder finder
# Walks the extracted tree and finds the folder that contains
# subfolders with actual image files inside
DATASET_DIR = None
IMG_EXTS    = ('.pgm', '.jpg', '.jpeg', '.png', '.bmp')

for root, dirs, files in os.walk('extracted_dataset'):
    dirs.sort()
    subfolders_with_imgs = []
    for d in sorted(dirs):
        dpath = os.path.join(root, d)
        imgs  = [f for f in os.listdir(dpath) if f.lower().endswith(IMG_EXTS)]
        if imgs:
            subfolders_with_imgs.append(d)
    if len(subfolders_with_imgs) >= 5:
        DATASET_DIR = root
        break

if DATASET_DIR is None:
    # Last resort: manually scan for any .pgm/.jpg files
    for root, dirs, files in os.walk('extracted_dataset'):
        imgs = [f for f in files if f.lower().endswith(IMG_EXTS)]
        if imgs:
            DATASET_DIR = os.path.dirname(root)
            print(f'Found images in: {root}')
            break

if DATASET_DIR is None:
    raise FileNotFoundError(
        'Could not find image subfolders in ZIP.\n'
        'Please set DATASET_DIR manually to the folder containing s1/, s2/, ... subfolders.'
    )

subjects = sorted([d for d in os.listdir(DATASET_DIR)
                   if os.path.isdir(os.path.join(DATASET_DIR, d))])
total_imgs = sum(
    len([f for f in os.listdir(os.path.join(DATASET_DIR, s))
         if f.lower().endswith(IMG_EXTS)])
    for s in subjects
)

print(f'\nDataset folder : {DATASET_DIR}')
print(f'Subjects found : {len(subjects)}')
print(f'Total images   : {total_imgs}')
print(f'First few      : {subjects[:5]}')


---
## STEP 3 — Build Face Database (mn x p)

Each image of size m x n is **flattened into a column vector** of length mn.  
With p total images, the face database shape is (mn x p).

In [ ]:
def load_face_database(dataset_path, img_size=IMG_SIZE):
    images, labels, subject_names = [], [], []
    IMG_EXTS = ('.pgm', '.jpg', '.jpeg', '.png', '.bmp')

    subject_dirs = sorted([
        d for d in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, d))
    ])

    if not subject_dirs:
        raise ValueError(f'No subfolders found in: {dataset_path}')

    for label_idx, subject in enumerate(subject_dirs):
        subject_names.append(subject)
        subj_path = os.path.join(dataset_path, subject)
        img_files = sorted([
            f for f in os.listdir(subj_path)
            if f.lower().endswith(IMG_EXTS)
        ])
        if not img_files:
            print(f'  Warning: No images in {subj_path} — skipping')
            continue
        for fname in img_files:
            img = cv2.imread(os.path.join(subj_path, fname), cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f'  Warning: Could not read {fname} — skipping')
                continue
            img = cv2.resize(img, img_size)
            images.append(img.flatten().astype(np.float64))
            labels.append(label_idx)

    if not images:
        raise ValueError(
            f'No images loaded from {dataset_path}!\n'
            f'Subfolders found: {subject_dirs[:5]}\n'
            f'Please check the DATASET_DIR path is correct and contains image files.'
        )

    print(f'Loaded {len(images)} images from {len(subject_names)} subjects.')
    Face_Db = np.column_stack(images)  # (mn, p)
    return Face_Db, np.array(labels), subject_names


Face_Db, labels, subject_names = load_face_database(DATASET_DIR)
mn, p         = Face_Db.shape
num_subjects  = len(subject_names)

print('=' * 45)
print(f'  Face_Db shape : {Face_Db.shape}  (mn x p)')
print(f'  Image size    : {IMG_SIZE[1]} x {IMG_SIZE[0]} = {mn} pixels')
print(f'  Total images  : {p}')
print(f'  Subjects      : {num_subjects}')
print(f'  Imgs/subject  : {p // num_subjects}')
print('=' * 45)


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13, 6))
fig.suptitle('Sample Faces from Database', fontsize=14, fontweight='bold')
shown, ax_flat = set(), iter(axes.flat)
for i in range(p):
    if labels[i] not in shown:
        ax = next(ax_flat, None)
        if ax is None: break
        ax.imshow(Face_Db[:, i].reshape(IMG_SIZE[1], IMG_SIZE[0]), cmap='gray')
        ax.set_title(f'Subject {subject_names[labels[i]]}', fontsize=9)
        ax.axis('off')
        shown.add(labels[i])
plt.tight_layout()
plt.show()

---
## STEP 4 — PCA: Eigenfaces (Turk & Pentland, 1991)

| Step | Formula | Shape |
|------|---------|-------|
| Mean Face | M = mean(Face_Db) | mn x 1 |
| Mean-Zero | Phi = Face_Db - M | mn x p |
| Surrogate Covariance | C = Phi.T @ Phi | p x p |
| Eigen decomp | eigh(C) | eigenvalues, eigenvectors |
| Feature Vectors | Top-k eigenvectors V | p x k |
| Eigenfaces | Psi = V.T @ Phi.T | k x mn |
| Signatures | Omega = Psi @ Phi | k x p |

In [ ]:
class PCAFaceRecognizer:
    """PCA Eigenfaces — Turk & Pentland (1991). NumPy + SciPy only."""

    def __init__(self, k=50):
        self.k = k
        self.M = self.Phi = self.Psi = self.Omega = self.eigenvalues = None

    def fit(self, Face_Db):
        mn, p = Face_Db.shape

        # Step 2: Mean Face (mn x 1)
        self.M   = np.mean(Face_Db, axis=1, keepdims=True)

        # Step 3: Mean-Zero Faces (mn x p)
        self.Phi = Face_Db - self.M

        # Step 4: Surrogate Covariance C = Phi.T @ Phi  (p x p)
        C = self.Phi.T @ self.Phi

        # Step 5: Eigendecomposition
        eigenvalues, eigenvectors = eigh(C)             # ascending
        idx           = np.argsort(eigenvalues)[::-1]   # sort descending
        self.eigenvalues = eigenvalues[idx]
        eigenvectors  = eigenvectors[:, idx]             # (p x p)

        # Step 6: Top-k Feature Vectors (p x k)
        V = eigenvectors[:, :self.k]

        # Step 7: Eigenfaces Psi = V.T @ Phi.T  (k x mn)
        self.Psi = V.T @ self.Phi.T
        norms = np.linalg.norm(self.Psi, axis=1, keepdims=True)
        norms[norms == 0] = 1
        self.Psi /= norms

        # Step 8: Signatures Omega = Psi @ Phi  (k x p)
        self.Omega = self.Psi @ self.Phi

        print(f'PCA done | k={self.k} | Eigenfaces: {self.Psi.shape} | Signatures: {self.Omega.shape}')

    def transform(self, images):
        """Project images (mn x n) into eigenface space -> (k x n)"""
        return self.Psi @ (images - self.M)

    def variance_explained(self):
        return np.cumsum(self.eigenvalues) / np.sum(self.eigenvalues) * 100


pca = PCAFaceRecognizer(k=50)
pca.fit(Face_Db)

# Scree / variance plot
cumvar = pca.variance_explained()
plt.figure(figsize=(9, 4))
plt.plot(range(1, len(cumvar)+1), cumvar, 'b-', linewidth=1.5)
plt.fill_between(range(1, len(cumvar)+1), cumvar, alpha=0.1, color='blue')
plt.axhline(95, color='red',    linestyle='--', label='95% variance')
plt.axhline(99, color='orange', linestyle='--', label='99% variance')
plt.xlabel('Number of Components (k)'); plt.ylabel('Cumulative Variance Explained (%)')
plt.title('Scree Plot — PCA Variance Explained', fontweight='bold')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

print(f'k needed for 95% variance: {np.searchsorted(cumvar, 95)+1}')
print(f'k needed for 99% variance: {np.searchsorted(cumvar, 99)+1}')

In [ ]:
fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(2, 6, figure=fig)

ax_m = fig.add_subplot(gs[:, 0])
ax_m.imshow(pca.M.reshape(IMG_SIZE[1], IMG_SIZE[0]), cmap='gray')
ax_m.set_title('Mean Face', fontweight='bold'); ax_m.axis('off')

for i in range(10):
    ax = fig.add_subplot(gs[i//5, (i%5)+1])
    ax.imshow(pca.Psi[i].reshape(IMG_SIZE[1], IMG_SIZE[0]), cmap='gray')
    ax.set_title(f'EF {i+1}', fontsize=8); ax.axis('off')

fig.suptitle('Mean Face and Top 10 Eigenfaces', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## STEP 5 — Train / Test Split (60% / 40%)

In [ ]:
Omega_all = pca.Omega.T  # (p, k)

X_train, X_test, y_train, y_test = train_test_split(
    Omega_all, labels, test_size=0.4, random_state=42, stratify=labels)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train : {X_train.shape[0]} samples ({X_train.shape[0]/p*100:.0f}%)')
print(f'Test  : {X_test.shape[0]}  samples ({X_test.shape[0]/p*100:.0f}%)')
print(f'Dim   : {X_train.shape[1]} features (= k)')
print(f'Classes: {num_subjects}')

---
## STEP 6 — ANN: Backpropagation (NumPy only)

Architecture: Input(k) -> Hidden1(128, ReLU) -> Hidden2(64, ReLU) -> Output(num_subjects, Softmax)  
Optimizer: Mini-batch gradient descent with momentum  
Loss: Cross-entropy

In [ ]:
class ANN:
    """Feedforward Neural Network with Backpropagation. NumPy only."""

    def __init__(self, layer_sizes, lr=0.01, epochs=300, momentum=0.9, batch_size=16):
        self.layer_sizes  = layer_sizes
        self.lr           = lr
        self.epochs       = epochs
        self.momentum     = momentum
        self.batch_size   = batch_size
        self.loss_history = []
        self.acc_history  = []
        self._init_weights()

    def _init_weights(self):
        self.W = []
        self.b = []
        for i in range(len(self.layer_sizes) - 1):
            scale = np.sqrt(2.0 / self.layer_sizes[i])
            self.W.append(np.random.randn(self.layer_sizes[i], self.layer_sizes[i+1]) * scale)
            self.b.append(np.zeros((1, self.layer_sizes[i+1])))

    @staticmethod
    def relu(z):       return np.maximum(0, z)
    @staticmethod
    def relu_grad(z):  return (z > 0).astype(float)
    @staticmethod
    def softmax(z):
        e = np.exp(z - np.max(z, axis=1, keepdims=True))
        return e / np.sum(e, axis=1, keepdims=True)

    def _forward(self, X):
        A, As, Zs = X, [X], []
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            Z = A @ W + b
            Zs.append(Z)
            A = self.softmax(Z) if i == len(self.W)-1 else self.relu(Z)
            As.append(A)
        return As, Zs

    @staticmethod
    def _loss(Y_pred, Y_oh):
        return -np.mean(np.sum(Y_oh * np.log(Y_pred + 1e-12), axis=1))

    def _backward(self, As, Zs, Y_oh):
        n  = Y_oh.shape[0]
        dW = [None]*len(self.W)
        db = [None]*len(self.b)
        dZ = As[-1] - Y_oh
        for i in reversed(range(len(self.W))):
            dW[i] = As[i].T @ dZ / n
            db[i] = np.mean(dZ, axis=0, keepdims=True)
            if i > 0:
                dZ = (dZ @ self.W[i].T) * self.relu_grad(Zs[i-1])
        return dW, db

    def fit(self, X, y, X_val=None, y_val=None, verbose=True):
        n_cls  = self.layer_sizes[-1]
        Y_oh   = np.zeros((len(y), n_cls))
        Y_oh[np.arange(len(y)), y] = 1
        vW = [np.zeros_like(w) for w in self.W]
        vb = [np.zeros_like(b) for b in self.b]
        n  = X.shape[0]

        for epoch in range(self.epochs):
            perm = np.random.permutation(n)
            Xs, Ys = X[perm], Y_oh[perm]
            ep_loss, nb = 0, 0
            for s in range(0, n, self.batch_size):
                Xb, Yb      = Xs[s:s+self.batch_size], Ys[s:s+self.batch_size]
                As, Zs      = self._forward(Xb)
                ep_loss    += self._loss(As[-1], Yb)
                nb         += 1
                dW, db      = self._backward(As, Zs, Yb)
                for i in range(len(self.W)):
                    vW[i] = self.momentum * vW[i] - self.lr * dW[i]
                    vb[i] = self.momentum * vb[i] - self.lr * db[i]
                    self.W[i] += vW[i]
                    self.b[i] += vb[i]

            avg_loss = ep_loss / nb
            tr_acc   = accuracy_score(y, self.predict(X))
            self.loss_history.append(avg_loss)
            self.acc_history.append(tr_acc * 100)

            if verbose and (epoch+1) % 50 == 0:
                vs = ''
                if X_val is not None:
                    vs = f'  |  Val: {accuracy_score(y_val, self.predict(X_val))*100:.1f}%'
                print(f'Epoch [{epoch+1:>3}/{self.epochs}]  Loss: {avg_loss:.4f}  Train: {tr_acc*100:.1f}%{vs}')

    def predict_proba(self, X): return self._forward(X)[0][-1]
    def predict(self, X):       return np.argmax(self.predict_proba(X), axis=1)


print('ANN class ready!')

---
## STEP 7 — Train ANN and Evaluate

In [ ]:
k = 50

ann = ANN(
    layer_sizes = [k, 128, 64, num_subjects],
    lr=0.01, epochs=300, momentum=0.9, batch_size=16
)

print(f'Architecture: {k} -> 128 (ReLU) -> 64 (ReLU) -> {num_subjects} (Softmax)\n')
ann.fit(X_train, y_train, X_val=X_test, y_val=y_test)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(ann.loss_history, 'b-', linewidth=1.5)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss', fontweight='bold'); ax1.grid(True, alpha=0.3)

ax2.plot(ann.acc_history, 'g-', linewidth=1.5)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training Accuracy', fontweight='bold'); ax2.grid(True, alpha=0.3)

plt.suptitle('ANN Training Progress', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
y_pred = ann.predict(X_test)
acc    = accuracy_score(y_test, y_pred)

print('=' * 50)
print(f'  TEST ACCURACY  : {acc*100:.2f}%')
print(f'  k (components) : {k}')
print(f'  Split          : 60% train / 40% test')
print('=' * 50)
print()
print(classification_report(y_test, y_pred,
      target_names=[f'Subj_{s}' for s in subject_names]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(11, 9))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xlabel('Predicted Label', fontsize=11)
plt.ylabel('True Label', fontsize=11)
plt.title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## STEP 8 — Experiment A: Accuracy vs k

Vary the number of PCA components k and see how classification accuracy changes.  
Plot a graph of **Accuracy vs k** as required by the project.

In [ ]:
def full_pipeline(Face_Db, labels, subject_names, k, epochs=200):
    pca_t = PCAFaceRecognizer(k=k)
    pca_t.fit(Face_Db)
    Xtr, Xte, ytr, yte = train_test_split(
        pca_t.Omega.T, labels, test_size=0.4, random_state=42, stratify=labels)
    sc  = StandardScaler()
    Xtr = sc.fit_transform(Xtr)
    Xte = sc.transform(Xte)
    mdl = ANN([k, 128, 64, len(subject_names)],
               lr=0.01, epochs=epochs, momentum=0.9, batch_size=16)
    mdl.fit(Xtr, ytr, verbose=False)
    return accuracy_score(yte, mdl.predict(Xte)) * 100


k_values   = [k for k in [5, 10, 20, 30, 50, 75, 100, 150, 200] if k <= p]
accuracies = []

print('Running Experiment A...\n')
for k_val in k_values:
    a = full_pipeline(Face_Db, labels, subject_names, k=k_val, epochs=200)
    accuracies.append(a)
    print(f'  k = {k_val:>4}  ->  Accuracy = {a:.2f}%')

print('\nExperiment A complete!')

In [ ]:
best_idx = int(np.argmax(accuracies))
best_k   = k_values[best_idx]
best_acc = accuracies[best_idx]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_values, accuracies, 'b-o', linewidth=2, markersize=7,
        markerfacecolor='white', markeredgewidth=2)
ax.plot(best_k, best_acc, 'r*', markersize=16,
        label=f'Best: k={best_k}, Acc={best_acc:.1f}%', zorder=5)
for kv, ac in zip(k_values, accuracies):
    ax.annotate(f'{ac:.1f}%', (kv, ac),
                textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=8.5)
ax.set_xlabel('Number of PCA Components (k)', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Experiment A — Accuracy vs k (Number of Eigenfaces)', fontsize=13, fontweight='bold')
ax.set_xticks(k_values); ax.set_ylim(0, 115)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Best k = {best_k}  ->  Accuracy = {best_acc:.2f}%')
print('Observation: Too few components = underfitting. Too many = noise included.')

---
## STEP 9 — Experiment B: Imposter Detection

Add imposters (subjects NOT in the training set) to the test set.  
The system should identify them as **Not Enrolled** using a confidence threshold.

- Confidence >= threshold -> Accept as enrolled subject
- Confidence < threshold  -> Reject as Imposter

In [ ]:
# Hold out last 5 subjects as imposters
N_IMP        = 5
imp_ids      = list(range(num_subjects - N_IMP, num_subjects))
enr_ids      = list(range(num_subjects - N_IMP))

enr_mask     = np.isin(labels, enr_ids)
imp_mask     = np.isin(labels, imp_ids)

Face_Db_enr  = Face_Db[:, enr_mask]
labels_enr   = LabelEncoder().fit_transform(labels[enr_mask])
Face_Db_imp  = Face_Db[:, imp_mask]
n_enr        = len(np.unique(labels_enr))

print(f'Enrolled subjects : {n_enr}')
print(f'Enrolled images   : {Face_Db_enr.shape[1]}')
print(f'Imposter subjects : {N_IMP}')
print(f'Imposter images   : {Face_Db_imp.shape[1]}')

In [ ]:
K_BEST  = best_k
pca_enr = PCAFaceRecognizer(k=K_BEST)
pca_enr.fit(Face_Db_enr)

Xtr_e, Xte_e, ytr_e, yte_e = train_test_split(
    pca_enr.Omega.T, labels_enr,
    test_size=0.4, random_state=42, stratify=labels_enr)

sc_enr = StandardScaler()
Xtr_e  = sc_enr.fit_transform(Xtr_e)
Xte_e  = sc_enr.transform(Xte_e)

ann_enr = ANN([K_BEST, 128, 64, n_enr],
               lr=0.01, epochs=300, momentum=0.9, batch_size=16)
print(f'Training on enrolled subjects only (k={K_BEST})...')
ann_enr.fit(Xtr_e, ytr_e, verbose=False)

enr_acc = accuracy_score(yte_e, ann_enr.predict(Xte_e))
print(f'Enrolled-only test accuracy: {enr_acc*100:.2f}%')

In [ ]:
# Project imposters into enrolled PCA space
omega_imp   = pca_enr.transform(Face_Db_imp).T
omega_imp_s = sc_enr.transform(omega_imp)

enr_conf = np.max(ann_enr.predict_proba(Xte_e),      axis=1)
imp_conf = np.max(ann_enr.predict_proba(omega_imp_s), axis=1)

print(f'Avg confidence — Enrolled : {np.mean(enr_conf):.4f}')
print(f'Avg confidence — Imposters: {np.mean(imp_conf):.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(enr_conf, bins=20, alpha=0.65, color='#3b82f6', label=f'Enrolled (n={len(enr_conf)})')
ax.hist(imp_conf, bins=20, alpha=0.65, color='#ef4444', label=f'Imposters (n={len(imp_conf)})')
ax.set_xlabel('Max Softmax Confidence', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Confidence Distribution: Enrolled vs Imposters', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Threshold sweep
thresholds   = np.arange(0.05, 1.0, 0.05)
enr_acc_list = []
imp_det_list = []
enr_preds    = ann_enr.predict(Xte_e)

for thresh in thresholds:
    n_correct  = np.sum((enr_preds == yte_e) & (enr_conf >= thresh))
    n_detected = np.sum(imp_conf < thresh)
    enr_acc_list.append(n_correct  / len(yte_e)  * 100)
    imp_det_list.append(n_detected / len(imp_conf) * 100)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, enr_acc_list, 'b-o', markersize=5, linewidth=2, label='Enrolled Recognition Rate')
ax.plot(thresholds, imp_det_list, 'r-s', markersize=5, linewidth=2, label='Imposter Detection Rate')
ax.set_xlabel('Confidence Threshold', fontsize=11)
ax.set_ylabel('Rate (%)', fontsize=11)
ax.set_title('Experiment B — Recognition Rate vs Imposter Detection Rate', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

eer_idx = int(np.argmin(np.abs(np.array(enr_acc_list) - np.array(imp_det_list))))
print(f'EER threshold approx: {thresholds[eer_idx]:.2f}')
print(f'At EER -> Recognition: {enr_acc_list[eer_idx]:.1f}%  |  Imposter Detection: {imp_det_list[eer_idx]:.1f}%')

---
## BONUS — Face Reconstruction at Different k Values

In [ ]:
test_img = Face_Db[:, 0:1]
k_list   = [k for k in [5, 10, 20, 50, 100, 200] if k <= p]

fig, axes = plt.subplots(1, len(k_list)+1, figsize=(15, 4))
axes[0].imshow(test_img.reshape(IMG_SIZE[1], IMG_SIZE[0]), cmap='gray')
axes[0].set_title('Original', fontweight='bold'); axes[0].axis('off')

for i, k_val in enumerate(k_list):
    pca_t = PCAFaceRecognizer(k=k_val)
    pca_t.fit(Face_Db)
    sig   = pca_t.Psi @ (test_img - pca_t.M)
    recon = np.clip(pca_t.Psi.T @ sig + pca_t.M, 0, 255)
    axes[i+1].imshow(recon.reshape(IMG_SIZE[1], IMG_SIZE[0]), cmap='gray')
    axes[i+1].set_title(f'k = {k_val}', fontsize=10)
    axes[i+1].axis('off')

fig.suptitle('Face Reconstruction Quality vs k', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Summary

| Item | Detail |
|------|--------|
| Dataset | AT&T/ORL — 40 subjects x 10 images = 400 total |
| Image size | 92 x 112 px (mn = 10,304) |
| PCA | Surrogate covariance — Turk & Pentland (1991) |
| ANN | Backpropagation — NumPy only |
| Architecture | Input -> 128 (ReLU) -> 64 (ReLU) -> Softmax |
| Split | 60% Train / 40% Test |
| Experiment A | Accuracy vs k — finds optimal PCA components |
| Experiment B | Imposter detection via confidence thresholding |

### Libraries used (as per project spec)
- `numpy` — all matrix operations and ANN
- `scipy.linalg.eigh` — eigenvalue/eigenvector decomposition
- `cv2` (OpenCV) — image reading and resizing